In [2]:
import pandas as pd

df = pd.read_csv('orders.csv')

# Data Understanding

In [3]:
df.head()

,order_id,customer_id,order_date,total_amount,category,city
0,825,272,05-01-2024,18832,Accessories,Mumbai
1,694,229,05-01-2024,2546,Accessories,Mumbai
2,267,85,07-01-2024,5545,Furniture,Pune
3,215,70,09-01-2024,54986,Accessories,Chennai
4,163,54,12-01-2024,47293,Electronics,Pune


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 890 entries, 0 to 889
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   order_id      890 non-null    int64 
 1   customer_id   890 non-null    int64 
 2   order_date    890 non-null    object
 3   total_amount  890 non-null    int64 
 4   category      890 non-null    object
 5   city          890 non-null    object
dtypes: int64(3), object(3)
memory usage: 41.8+ KB


In [5]:
df.describe()

,order_id,customer_id,total_amount
count,890.000000,890.000000,890.000000
mean,445.500000,146.166292,19574.428090
std,257.065167,85.535864,17274.811966
min,1.000000,1.000000,504.000000
25%,223.250000,71.000000,5615.250000
50%,445.500000,148.000000,12712.500000
75%,667.750000,223.750000,32139.750000
max,890.000000,300.000000,59936.000000


# Data Cleaning

In [6]:
df['order_date'] = pd.to_datetime(df['order_date'], format = '%d-%m-%Y')

In [7]:
df.isnull().sum()

order_id        0
customer_id     0
order_date      0
total_amount    0
category        0
city            0
dtype: int64

In [8]:
df.drop_duplicates(inplace = True)

# Trend Analysis

In [9]:
df['Month'] = df['order_date'].dt.to_period('M')

Trend = df.groupby('Month')['total_amount'].sum()
Trend

Month
2024-01     491570
2024-02     675964
2024-03    1515729
2024-04    2127895
2024-05    1963598
2024-06    2073671
2024-07    1512361
2024-08    2140123
2024-09    2373024
2024-10    2547306
Freq: M, Name: total_amount, dtype: int64

# Segmentation 

In [10]:
cust_segment = df.groupby('customer_id').agg({
    'order_id' : 'count',                             # count number of orders
    'total_amount' : 'sum'                            # total spending
}).reset_index()


cust_segment = cust_segment.rename(columns = {
    'order_id' : 'Total_orders',
    'total_amount':'total_spent'
}
)

cust_segment

,customer_id,Total_orders,total_spent
0,1,3,39997
1,2,4,37179
2,3,2,8813
3,4,4,109651
4,5,5,238658
...,...,...,...
289,296,2,6308
290,297,1,4962
291,298,2,18155
292,299,2,4918


In [11]:
cust_segment['total_spent'].mean()

np.float64(59255.921768707485)

# Customer Segmentation Using Median-Based Approach

In [12]:
median_cutoff = cust_segment['total_spent'].median()
median_cutoff

np.float64(14135.5)

In [13]:
cust_segment['segment'] = cust_segment['total_spent'].apply(
    lambda x: 'High Value' if x >= median_cutoff else 'Low Value'
)

In [14]:
cust_segment['segment'].value_counts()

segment
High Value    147
Low Value     147
Name: count, dtype: int64

In [15]:
Revenue_contribution = cust_segment.groupby('segment')['total_spent'].sum()
Revenue_contribution

segment
High Value    16830365
Low Value       590876
Name: total_spent, dtype: int64

# High-Value Customer Identification Using Top 20% Approach

In [16]:
top_20_cutoff = cust_segment['total_spent'].quantile(0.80)
top_20_cutoff

np.float64(52388.80000000001)

In [17]:
cust_segment['segement_top20'] = cust_segment['total_spent'].apply(
    lambda x : 'High Value' if x >= top_20_cutoff else 'Low Value'
)

In [18]:
cust_segment

,customer_id,Total_orders,total_spent,segment,segement_top20
0,1,3,39997,High Value,Low Value
1,2,4,37179,High Value,Low Value
2,3,2,8813,Low Value,Low Value
3,4,4,109651,High Value,High Value
4,5,5,238658,High Value,High Value
...,...,...,...,...,...
289,296,2,6308,Low Value,Low Value
290,297,1,4962,Low Value,Low Value
291,298,2,18155,High Value,Low Value
292,299,2,4918,Low Value,Low Value


In [19]:
cust_segment['segement_top20'].value_counts()

segement_top20
Low Value     235
High Value     59
Name: count, dtype: int64

In [20]:
Revenue_contribution_top20 = cust_segment.groupby('segement_top20')['total_spent'].sum()
Revenue_contribution_top20

segement_top20
High Value    14022869
Low Value      3398372
Name: total_spent, dtype: int64

# Cohort Analysis

In [21]:
df['Cohort Month'] = df.groupby('customer_id')['order_date'].transform('min').dt.to_period('M')

In [22]:
df['Month_Number'] = (df['order_date'].dt.to_period('M') - df['Cohort Month']).apply(lambda x : x.n)

In [23]:
df

,order_id,customer_id,order_date,total_amount,category,city,Month,Cohort Month,Month_Number
0,825,272,2024-01-05,18832,Accessories,Mumbai,2024-01,2024-01,0
1,694,229,2024-01-05,2546,Accessories,Mumbai,2024-01,2024-01,0
2,267,85,2024-01-07,5545,Furniture,Pune,2024-01,2024-01,0
3,215,70,2024-01-09,54986,Accessories,Chennai,2024-01,2024-01,0
4,163,54,2024-01-12,47293,Electronics,Pune,2024-01,2024-01,0
...,...,...,...,...,...,...,...,...,...
885,699,233,2024-10-30,7808,Electronics,Chennai,2024-10,2024-05,5
886,103,37,2024-10-30,42712,Furniture,Delhi,2024-10,2024-02,8
887,95,32,2024-10-30,4012,Electronics,Delhi,2024-10,2024-10,0
888,317,100,2024-10-31,5988,Furniture,Mumbai,2024-10,2024-04,6


In [25]:
cohort = df.groupby(['Cohort Month', 'Month_Number'])['customer_id'].nunique().reset_index()

In [26]:
cohort

,Cohort Month,Month_Number,customer_id
0,2024-01,0,16
1,2024-01,1,5
2,2024-01,2,6
3,2024-01,3,2
4,2024-01,4,5
5,2024-01,5,7
6,2024-01,6,4
7,2024-01,7,3
8,2024-01,8,5
9,2024-01,9,6


# Customer Retention Analysis

In [31]:
cust_level = df.groupby(['city','customer_id']).agg({
    'order_id' : 'count'
}).reset_index()

cust_level.rename(columns={
    'order_id' : 'Total_orders'
}, inplace = True)

cust_level['is_repeat'] = cust_level['Total_orders'] > 1
cust_level

,city,customer_id,Total_orders,is_repeat
0,Bangalore,2,2,True
1,Bangalore,5,1,False
2,Bangalore,7,1,False
3,Bangalore,8,1,False
4,Bangalore,9,1,False
...,...,...,...,...
642,Pune,279,1,False
643,Pune,283,1,False
644,Pune,290,1,False
645,Pune,293,1,False


In [32]:
city_retention = cust_level.groupby('city').agg({
        'customer_id' : 'nunique',
        'is_repeat' : 'sum'}
).reset_index()

city_retention.rename(columns = {
    'customer_id': 'total_customers',
    'is_repeat': 'repeat_customers'
}, inplace = True)

city_retention

,city,total_customers,repeat_customers
0,Bangalore,125,39
1,Chennai,128,38
2,Delhi,133,31
3,Mumbai,129,32
4,Pune,132,37


In [35]:
city_retention['retention_rate'] = round(city_retention['repeat_customers'] * 100 / city_retention['total_customers'],2)
city_retention

,city,total_customers,repeat_customers,retention_rate
0,Bangalore,125,39,31.20
1,Chennai,128,38,29.69
2,Delhi,133,31,23.31
3,Mumbai,129,32,24.81
4,Pune,132,37,28.03


# Funnel Analysis

In [79]:
Customers_with_orders = df['customer_id'].nunique()

In [31]:
total_orders = df['order_id'].nunique()
total_orders

890

In [33]:
High_Value_Orders_Greater_than_20K = df[df['total_amount'] > 20000]['order_id'].nunique()
High_Value_Orders_Greater_than_20K

331

In [29]:
Very_High_Value_Orders_Greater_than_30K = df[df['total_amount'] > 30000]['order_id'].nunique()
Very_High_Value_Orders_Greater_than_30K

241

In [65]:
funnel1 = pd.DataFrame({
    'Stage' : ['Total Orders', 'Orders > ₹20K', 'Orders > ₹30K'],
    'Count' : [total_orders, High_Value_Orders_Greater_than_20K, Very_High_Value_Orders_Greater_than_30K ]
})

In [66]:
funnel1

,Stage,Count
0,Total Orders,890
1,Orders > ₹20K,331
2,Orders > ₹30K,241
